# Satellite Collision Prediction — RetNet vs Transformer
## Colab Training Notebook (T4 GPU)

Run cells top-to-bottom. ~5-10 min on T4.

In [ ]:
#@title 1. Setup
!pip install -q torch scikit-learn tqdm pandas
import torch
print(f'PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available(): print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
#@title 2. Upload cdms.csv
from google.colab import files
import os
uploaded = files.upload()
csv_name = list(uploaded.keys())[0]
os.makedirs('data', exist_ok=True)
!mv {csv_name} data/cdms.csv
print(f'Uploaded -> data/cdms.csv')

In [ ]:
#@title 3. Write source files
import os
os.makedirs('src', exist_ok=True)
os.makedirs('data/processed', exist_ok=True)

# --- data_processing.py ---
with open('src/data_processing.py','w') as f: f.write('''
import os, numpy as np, pandas as pd, torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

SEQ_LEN=25; PC_THRESHOLD=1e-4; RANDOM_SEED=42; N_TIME_FEATURES=5
NUMERIC_FEATURE_COLS=['MIN_RNG','PC','SAT1_RCS','SAT2_RCS','SAT_1_EXCL_VOL','SAT_2_EXCL_VOL','EMERGENCY_REPORTABLE']
CATEGORICAL_COLS=['SAT1_OBJECT_TYPE','SAT2_OBJECT_TYPE']

def load_cdms(p):
    print(f'Loading {p}...'); df=pd.read_csv(p); print(f'Shape: {df.shape}')
    df.columns=[c.strip().upper() for c in df.columns]
    df['TCA']=pd.to_datetime(df['TCA'],errors='coerce'); df=df.dropna(subset=['TCA'])
    df['CREATED']=pd.to_datetime(df['CREATED'],errors='coerce'); df=df.sort_values('TCA').reset_index(drop=True)
    print(f'Rows: {len(df)}, PC: {df["PC"].min():.2e} to {df["PC"].max():.2e}')
    nhr=(df['PC']>PC_THRESHOLD).sum(); print(f'High-risk: {nhr}/{len(df)} ({100*nhr/len(df):.1f}%)')
    return df

def create_groups(df,wh=168.0):
    gs={}; df=df.sort_values(['SAT_1_ID','SAT_2_ID','TCA']).reset_index(drop=True); gid=0
    for (o1,o2),pdf in df.groupby(['SAT_1_ID','SAT_2_ID']):
        pdf=pdf.sort_values('TCA').reset_index(drop=True); si=0
        for i in range(1,len(pdf)):
            if (pdf.iloc[i]['TCA']-pdf.iloc[i-1]['TCA']).total_seconds()/3600>wh:
                gs[f'{o1}_{o2}_{gid}']=pdf.iloc[si:i].copy(); gid+=1; si=i
        gs[f'{o1}_{o2}_{gid}']=pdf.iloc[si:].copy(); gid+=1
    print(f'Groups: {len(gs)}'); return gs

def tfeats(gdf):
    n=len(gdf); f=np.zeros((n,N_TIME_FEATURES),dtype=np.float32)
    tca=gdf['TCA'].values.astype('datetime64[us]'); cre=gdf['CREATED'].values.astype('datetime64[us]')
    h2t=(tca-cre).astype('timedelta64[m]').astype(np.float32)/60
    f[:,0]=h2t; f[:,1]=(cre-cre[0]).astype('timedelta64[m]').astype(np.float32)/60
    f[:,2]=pd.to_datetime(tca).hour.astype(np.float32)
    f[:,3]=pd.to_datetime(tca).dayofweek.astype(np.float32); f[:,4]=h2t/24
    return f

def g2seq(gdf,fc,sl=SEQ_LEN):
    nd=np.zeros((len(gdf),len(fc)),dtype=np.float32)
    for i,c in enumerate(fc):
        if c in gdf.columns: nd[:,i]=pd.to_numeric(gdf[c],errors='coerce').fillna(0).values.astype(np.float32)
    cd=np.zeros((len(gdf),len(CATEGORICAL_COLS)),dtype=np.float32)
    for i,c in enumerate(CATEGORICAL_COLS):
        if c in gdf.columns: cd[:,i]=pd.to_numeric(gdf[c],errors='coerce').fillna(-1).values.astype(np.float32)
    af=np.concatenate([nd,cd,tfeats(gdf)],axis=-1); n=af.shape[0]
    seq=af[-sl:] if n>=sl else np.concatenate([np.zeros((sl-n,af.shape[1]),dtype=np.float32),af],axis=0)
    pc=pd.to_numeric(gdf['PC'],errors='coerce').fillna(0).max()
    return seq,(1.0 if pc>PC_THRESHOLD else 0.0)

def build_dataset(csv_p='data/cdms.csv',out='data/processed'):
    df=load_cdms(csv_p); fc=NUMERIC_FEATURE_COLS; nc=len(CATEGORICAL_COLS)
    nt=len(fc)+nc+N_TIME_FEATURES; print(f'Features/timestep: {nt}')
    gs=create_groups(df); seqs,labs=[],[]
    for gid,gdf in gs.items():
        if len(gdf)<1: continue
        s,l=g2seq(gdf,fc); seqs.append(s); labs.append(l)
    X=np.stack(seqs); y=np.array(labs,dtype=np.float32)
    print(f'Dataset: {X.shape}, High-risk: {int(y.sum())}/{len(y)} ({100*y.mean():.1f}%)')
    if y.sum()==0: print('WARNING: No high-risk!'); return
    sc=StandardScaler(); X=sc.fit_transform(X.reshape(-1,X.shape[-1])).reshape(X.shape)
    idx=np.arange(len(X)); ti,te=train_test_split(idx,train_size=0.7,random_state=RANDOM_SEED,stratify=y)
    vi,tei=train_test_split(te,train_size=0.15/0.3,random_state=RANDOM_SEED,stratify=y[te])
    os.makedirs(out,exist_ok=True)
    torch.save((torch.FloatTensor(X[ti]),torch.FloatTensor(y[ti])),os.path.join(out,'train.pt'))
    torch.save((torch.FloatTensor(X[vi]),torch.FloatTensor(y[vi])),os.path.join(out,'val.pt'))
    torch.save((torch.FloatTensor(X[tei]),torch.FloatTensor(y[tei])),os.path.join(out,'test.pt'))
    torch.save(sc,os.path.join(out,'scaler.pt'))
    print(f'Saved train:{len(ti)} val:{len(vi)} test:{len(tei)}')
    np_=int(y[ti].sum()); nn_=len(ti)-np_
    print(f'Class: {nn_} neg, {np_} pos | Focal alpha: {nn_/(nn_+np_):.4f}')
''')

# --- model.py ---
with open('src/model.py','w') as f: f.write('''
import torch, torch.nn as nn, torch.nn.functional as F
from typing import List, Optional

class MultiScaleRetention(nn.Module):
    def __init__(self,d,h,g=None,dp=0.1):
        super().__init__(); assert d%h==0
        self.d=d; self.h=h; self.dh=d//h; self.drop=nn.Dropout(dp)
        self.qkv=nn.Linear(d,3*d,bias=False); self.proj=nn.Linear(d,d,bias=False)
        if g is None: g=[0.9,0.95,0.99,0.995]
        if len(g)<h: g=g*(h//len(g)+1)
        self.register_buffer('gammas',torch.tensor(g[:h],dtype=torch.float32))
        self.ug=torch.unique(self.gammas)
    def _decay(self,T,gam):
        p=torch.arange(T,device=self.gammas.device,dtype=torch.float32); l=p.unsqueeze(1)-p.unsqueeze(0)
        m=torch.tril(torch.ones(T,T,device=self.gammas.device))
        d=torch.where(l==0,torch.ones_like(l),torch.clamp(l,min=1).pow(-gam))*m
        return d/d.sum(-1,keepdim=True).clamp(min=1e-6)
    def forward(self,x):
        B,T,C=x.shape; q,k,v=self.qkv(x).chunk(3,dim=-1)
        q=q.view(B,T,self.h,self.dh).transpose(1,2); k=k.view(B,T,self.h,self.dh).transpose(1,2); v=v.view(B,T,self.h,self.dh).transpose(1,2)
        o=torch.zeros_like(v)
        for g in self.ug:
            d=self._decay(T,g.item()); hi=(self.gammas==g).nonzero(as_tuple=True)[0]
            if len(hi)==0: continue
            qg,kg,vg=q[:,hi],k[:,hi],v[:,hi]; wkv=torch.einsum('ts,bhsd->bhtd',d,kg*vg); o[:,hi]=qg*wkv
        o=o.transpose(1,2).contiguous().view(B,T,C); return self.drop(self.proj(o))

class SwiGLU(nn.Module):
    def __init__(self,d,dp=0.1):
        super().__init__(); df=int(d*4/3); df=((df+1)//2)*2
        self.w1=nn.Linear(d,df,bias=False); self.w2=nn.Linear(df,d,bias=False); self.w3=nn.Linear(d,df,bias=False); self.drop=nn.Dropout(dp)
    def forward(self,x): return self.drop(self.w2(F.silu(self.w1(x))*self.w3(x)))

class RetNetBlock(nn.Module):
    def __init__(self,d,h,g=None,dp=0.1):
        super().__init__(); self.ret=MultiScaleRetention(d,h,g,dp); self.ffn=SwiGLU(d,dp)
        self.n1=nn.LayerNorm(d); self.n2=nn.LayerNorm(d)
    def forward(self,x): return x+self.ffn(self.n2(x+self.ret(self.n1(x))))

class RetNet(nn.Module):
    def __init__(self,idim,dm=128,nh=4,nl=4,g=None,dp=0.1):
        super().__init__(); self.inp=nn.Linear(idim,dm); self.pe=nn.Parameter(torch.randn(1,25,dm)*0.02)
        self.blks=nn.ModuleList([RetNetBlock(dm,nh,g,dp) for _ in range(nl)])
        self.norm=nn.LayerNorm(dm); self.cls=nn.Sequential(nn.Linear(dm,dm//2),nn.GELU(),nn.Dropout(dp),nn.Linear(dm//2,1))
        for m in self.modules():
            if isinstance(m,nn.Linear): nn.init.xavier_uniform_(m.weight)
    def forward(self,x):
        B,T,_=x.shape; x=self.inp(x)+self.pe[:,:T]
        for b in self.blks: x=b(x)
        return self.cls(self.norm(x)[:,-1,:])

class Transformer(nn.Module):
    def __init__(self,idim,dm=128,nh=4,nl=4,dff=256,dp=0.1):
        super().__init__(); self.inp=nn.Linear(idim,dm); self.pe=nn.Parameter(torch.randn(1,25,dm)*0.02)
        el=nn.TransformerEncoderLayer(dm,nh,dff,dp,activation='gelu',batch_first=True,norm_first=True)
        self.enc=nn.TransformerEncoder(el,nl,enable_nested_tensor=False)
        self.norm=nn.LayerNorm(dm); self.cls=nn.Sequential(nn.Linear(dm,dm//2),nn.GELU(),nn.Dropout(dp),nn.Linear(dm//2,1))
        for m in self.modules():
            if isinstance(m,nn.Linear): nn.init.xavier_uniform_(m.weight)
    def forward(self,x):
        B,T,_=x.shape; x=self.inp(x)+self.pe[:,:T]
        mask=nn.Transformer.generate_square_subsequent_mask(T,device=x.device)
        x=self.enc(x,mask=mask,is_causal=True); return self.cls(self.norm(x)[:,-1,:])
''')
print('Source files written!')

In [ ]:
#@title 4. Process data
import sys; sys.path.insert(0,'src')
from data_processing import build_dataset
build_dataset('data/cdms.csv')

In [ ]:
#@title 5. Train RetNet + Transformer
import sys; sys.path.insert(0,'src')
from model import RetNet, Transformer
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from tqdm import tqdm; import time

device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
X_tr,y_tr=torch.load('data/processed/train.pt',weights_only=False)
X_va,y_va=torch.load('data/processed/val.pt',weights_only=False)
idim=X_tr.shape[-1]; print(f'Input dim: {idim}, Device: {device}')

class FocalLoss(torch.nn.Module):
    def __init__(self,a=0.5,g=2.0): super().__init__(); self.a=a; self.g=g
    def forward(self,lo,tg):
        p=torch.sigmoid(lo.squeeze(-1)); tg=tg.float()
        bce=F.binary_cross_entropy(p,tg,reduction='none')
        pt=p*tg+(1-p)*(1-tg); return (self.a*tg+(1-self.a)*(1-tg))*(1-pt)**self.g*bce.mean()

def train_model(model,name,epochs=100,lr=1e-4,pat=15):
    model=model.to(device); opt=torch.optim.AdamW(model.parameters(),lr=lr,weight_decay=1e-4)
    crit=FocalLoss(); tl=DataLoader(TensorDataset(X_tr,y_tr),batch_size=32,shuffle=True,drop_last=True)
    vl=DataLoader(TensorDataset(X_va,y_va),batch_size=64)
    bf=0; be=0; hist={'tl':[],'vl':[],'vf':[]}
    for ep in range(1,epochs+1):
        model.train(); tloss=0
        for xb,yb in tqdm(tl,desc=f'{name} Ep{ep}',leave=False):
            xb,yb=xb.to(device),yb.to(device); opt.zero_grad()
            loss=crit(model(xb),yb); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); opt.step(); tloss+=loss.item()*xb.size(0)
        tloss/=len(tl.dataset)
        model.eval(); vp,vl_=[],[]; vloss=0
        with torch.no_grad():
            for xb,yb in vl:
                xb,yb=xb.to(device),yb.to(device); lo=model(xb).squeeze(-1)
                vloss+=F.binary_cross_entropy_with_logits(lo,yb,reduction='sum').item()
                vp.append(torch.sigmoid(lo).cpu().numpy()); vl_.append(yb.cpu().numpy())
        vloss/=len(vl.dataset); vp=np.concatenate(vp); vl_=np.concatenate(vl_)
        vf=f1_score(vl_,(vp>0.5).astype(int),zero_division=0)
        hist['tl'].append(tloss); hist['vl'].append(vloss); hist['vf'].append(vf)
        va=roc_auc_score(vl_,vp) if len(np.unique(vl_))>1 else 0
        print(f'Ep{ep:3d} TL:{tloss:.4f} VL:{vloss:.4f} F1:{vf:.4f} AUC:{va:.4f}')
        if vf>bf:
            bf=vf; be=ep
            torch.save({'ep':ep,'model_state_dict':model.state_dict(),'val_f1':vf,'model_name':name,'input_dim':idim},f'best_{name}.pt')
            print(f'  *** Best F1: {vf:.4f} ***')
        if ep-be>=pat: print(f'Early stop ep{ep}'); break
    return hist

print('=== Training RetNet ===')
retnet=RetNet(idim=idim); print(f'Params: {sum(p.numel() for p in retnet.parameters()):,}')
rh=train_model(retnet,'retnet')

print('\n=== Training Transformer ===')
trans=Transformer(idim=idim); print(f'Params: {sum(p.numel() for p in trans.parameters()):,}')
th=train_model(trans,'transformer')

In [ ]:
#@title 6. Evaluate + Plots
from sklearn.metrics import precision_score,recall_score,confusion_matrix,classification_report,roc_curve
import matplotlib.pyplot as plt; os.makedirs('results',exist_ok=True)

X_te,y_te=torch.load('data/processed/test.pt',weights_only=False)
tloader=DataLoader(TensorDataset(X_te,y_te),batch_size=64)

def evaluate(ckpt,cls,name):
    ck=torch.load(ckpt,map_location=device,weights_only=False); m=cls(idim=ck['input_dim']).to(device)
    m.load_state_dict(ck['model_state_dict']); m.eval()
    ps,ls=[],[]
    with torch.no_grad():
        for xb,yb in tloader:
            lo=m(xb.to(device)).squeeze(-1); ps.append(torch.sigmoid(lo).cpu().numpy()); ls.append(yb.numpy())
    ps=np.concatenate(ps); ls=np.concatenate(ls); pd_=(ps>0.5).astype(int)
    met={'Acc':accuracy_score(ls,pd_),'Prec':precision_score(ls,pd_,zero_division=0),
         'Rec':recall_score(ls,pd_,zero_division=0),'F1':f1_score(ls,pd_,zero_division=0),
         'AUC':roc_auc_score(ls,ps) if len(np.unique(ls))>1 else 0}
    cm=confusion_matrix(ls,pd_)
    print(f'\n{name.upper()}: {met}'); print(classification_report(ls,pd_,target_names=['Low','High']))
    return met,ps,ls,cm

rm,rp,tl,rcm=evaluate('best_retnet.pt',RetNet,'retnet')
tm,tp,_,tcm=evaluate('best_transformer.pt',Transformer,'transformer')

# Comparison table
fig,ax=plt.subplots(figsize=(10,3)); ax.axis('off')
rows=[[k,f'{rm[k]:.4f}',f'{tm[k]:.4f}','RetNet' if rm[k]>tm[k] else 'Transformer'] for k in rm]
tb=ax.table(cellText=rows,colLabels=['Metric','RetNet','Transformer','Winner'],cellLoc='center',loc='center')
tb.auto_set_font_size(False); tb.set_fontsize(12); tb.scale(1.2,1.8)
for j in range(4): tb[0,j].set_facecolor('#4472C4'); tb[0,j].set_text_props(color='white',fontweight='bold')
ax.set_title('RetNet vs Transformer',fontsize=14,pad=20); plt.tight_layout()
plt.savefig('results/comparison_table.png',bbox_inches='tight'); plt.show()

# ROC curves
fig,ax=plt.subplots(figsize=(7,6))
fpr1,tpr1,_=roc_curve(tl,rp); fpr2,tpr2,_=roc_curve(tl,tp)
ax.plot(fpr1,tpr1,'b-',lw=2,label=f'RetNet (AUC={rm["AUC"]:.4f})')
ax.plot(fpr2,tpr2,'r-',lw=2,label=f'Transformer (AUC={tm["AUC"]:.4f})')
ax.plot([0,1],[0,1],'k--'); ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
ax.set_title('ROC Curve'); ax.legend(); ax.grid(alpha=0.3)
plt.savefig('results/roc.png',bbox_inches='tight'); plt.show()

# Confusion matrices
for cm,n in [(rcm,'RetNet'),(tcm,'Transformer')]:
    fig,ax=plt.subplots(figsize=(5,4)); im=ax.imshow(cm,cmap=plt.cm.Blues)
    ax.set_title(f'CM - {n}'); plt.colorbar(im,ax=ax)
    ax.set_xticks([0,1]); ax.set_yticks([0,1]); ax.set_xticklabels(['Low','High']); ax.set_yticklabels(['Low','High'])
    for i in range(2):
        for j in range(2): ax.text(j,i,str(cm[i,j]),ha='center',va='center',fontsize=16,fontweight='bold',color='white' if cm[i,j]>cm.max()/2 else 'black')
    plt.tight_layout(); plt.savefig(f'results/{n}_cm.png',bbox_inches='tight'); plt.show()

print('All plots saved to results/')

In [ ]:
#@title 7. Download results
from google.colab import files
import glob
for f in glob.glob('results/*.png')+glob.glob('best_*.pt'):
    files.download(f)
print('Done!')